
# FoS v0.7.2 — Stage A → provisional trajectory Stage B → Stage C 전체 E2E

이 노트북은 새 Colab GPU 세션에서 다음 과정을 처음부터 수행한다.

1. Google Drive 연결
2. `FoS-feature-stage-c-v072-trajectory.zip` 압축 해제
3. 이전 `with_results.zip`에서 Stage A의 full evidence/ChEMBL cache만 복원
4. v0.7.2 설치 및 trajectory·Stage C 통합 테스트
5. Ollama 설치 및 `qwen3:8b` 실행
6. Heuristic A→B→C smoke test
7. Qwen3 8B 기반 실제 A→B→C trajectory 실행
8. Stage B 연속 경로와 Stage C 판정 점검
9. 결과를 즉시 Google Drive에 백업
10. 누적 `Δon × ΔS` objective-space PNG/GIF/MP4 생성

## 중요한 판정 기준

- `trajectory` 모드는 ACCEPT된 분자를 다음 단계의 유일한 parent로 사용한다.
- 이 모드에서는 실제 `BACKTRACK`이 없어야 한다.
- 경로는 `S0 → S1 → S2 ...`처럼 연속이어야 한다.
- 다만 통과 가능한 자식 후보가 없으면 `S0 → S1 → local_optimum`으로 끝날 수 있다.
- 외부 독립 QSAR 또는 docking 결과를 넣지 않으면 Stage C가 보수적으로
  `NEEDS_VALIDATION`을 반환하는 것이 정상일 수 있다.


## v0.7.2 이동 정책

- `ELIGIBLE`: 기존과 같이 이동 가능
- `PROVISIONAL`: trajectory mode에서만 제한적으로 이동 가능
- `NEEDS_VALIDATION`: 이동하지 않고 검증 대기
- Stage C는 독립 검증이 없으면 provisional tip을 `NEEDS_VALIDATION`으로 유지

기본 provisional 제한은 `step Δon ≥ -0.50`, `step ΔS ≥ +0.30`, `depth ≤ 2`, `uncertainty ≤ 0.70`, `cumulative Δon ≥ -0.75`이다.



## 0. Colab GPU 확인

Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 선택한 뒤 실행한다.
GPU가 없으면 Qwen3 8B가 CPU에서 실행되어 매우 느려지므로 이 셀에서 즉시 중단한다.


In [ ]:

import subprocess

gpu_check = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader",
    ],
    text=True,
    capture_output=True,
)

if gpu_check.returncode != 0:
    raise RuntimeError(
        "GPU가 연결되지 않았습니다. "
        "Colab 런타임을 GPU로 변경한 뒤 다시 시작하세요."
    )

print("Detected GPU:")
print(gpu_check.stdout)


## 1. Google Drive 연결

In [ ]:

from google.colab import drive

drive.mount("/content/drive")



## 2. 실행 경로와 설정

아래에서 **두 ZIP 경로만 본인의 Google Drive 위치에 맞게 수정**한다.

`WITH_RESULTS_ZIP`은 코드만 있는 ZIP이 아니라, 이전 full 실행의
`data/cache/evidence_live`가 포함된 ZIP이어야 한다.


In [ ]:

from pathlib import Path

# ============================================================
# 사용자가 수정할 두 경로
# ============================================================

V072_ZIP = Path(
    "/content/drive/MyDrive/FoS-feature-stage-c-v072-trajectory.zip"
)

WITH_RESULTS_ZIP = Path(
    "/content/drive/MyDrive/"
    "FoS-feature-stage-b-v2-implemented_with_results.zip"
)

# ============================================================
# 작업 및 결과 경로
# ============================================================

WORK_ROOT = Path("/content/fos_v072_e2e")
DRIVE_RUN_ROOT = Path(
    "/content/drive/MyDrive/FoS_v072_e2e_runs"
)

# ============================================================
# 실제 E2E 입력
# 이전 탐색에서 실측 improving analog가 확인된 CHEMBL285063
# ============================================================

SEED_COMPOUND_ID = "CHEMBL285063"
SEED_SMILES = (
    "C=CC(=O)Nc1ccc2ncnc("
    "Nc3cccc(Br)c3)c2c1"
)

ON_TARGET = "CHEMBL203"
OFF_TARGET = "CHEMBL1824"
PAIR_KEY = f"{ON_TARGET}__{OFF_TARGET}"

MODEL = "qwen3:8b"
OLLAMA_API_ROOT = "http://127.0.0.1:11434"
OLLAMA_BASE_URL = f"{OLLAMA_API_ROOT}/v1"

SEARCH_MODE = "trajectory"
MAX_ITERATIONS = 6
FINAL_TOP_K = 5

RUN_HEURISTIC_SMOKE = True
RUN_FOCUSED_TESTS = True

# 후보 하나의 Qwen E2E 최대 실행시간
QWEN_TIMEOUT_MINUTES = 45

# Stage C 외부 provider는 현재 기본적으로 비워둔다.
# 파일을 준비하면 Path(...)로 바꾼다.
PREDICTION_JSON = None
DOCKING_JSON = None
REQUIRE_DOCKING = False

print("v0.7.2 ZIP:", V072_ZIP, V072_ZIP.exists())
print(
    "with-results ZIP:",
    WITH_RESULTS_ZIP,
    WITH_RESULTS_ZIP.exists(),
)

if not V072_ZIP.exists():
    raise FileNotFoundError(
        f"v0.7.2 ZIP이 없습니다: {V072_ZIP}"
    )

if not WITH_RESULTS_ZIP.exists():
    raise FileNotFoundError(
        "이전 full cache가 든 with-results ZIP이 없습니다: "
        f"{WITH_RESULTS_ZIP}"
    )


## 3. v0.7.2 압축 해제 및 프로젝트 루트 자동 탐색

In [ ]:

import shutil
import zipfile


def find_v072_project_root(search_root: Path) -> Path:
    candidates = []

    for pyproject in search_root.rglob("pyproject.toml"):
        root = pyproject.parent

        required = [
            root / "src/stage_a",
            root / "src/stage_b",
            root / "src/stage_c",
            root / "scripts/run_stage_abc_live_pair.py",
            root / "TRAJECTORY_MODE.md",
        ]

        if all(path.exists() for path in required):
            candidates.append(root)

    if not candidates:
        raise FileNotFoundError(
            "압축 해제 결과에서 v0.7.2 프로젝트 루트를 "
            f"찾지 못했습니다: {search_root}"
        )

    candidates.sort(key=lambda path: len(path.parts))
    return candidates[0]


if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

V072_EXTRACT_ROOT = WORK_ROOT / "v072_extracted"
V072_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(V072_ZIP) as archive:
    archive.extractall(V072_EXTRACT_ROOT)

PROJECT_ROOT = find_v072_project_root(
    V072_EXTRACT_ROOT
)

print("PROJECT_ROOT:")
print(PROJECT_ROOT)

for relative in [
    "pyproject.toml",
    "src/stage_a/__init__.py",
    "src/stage_b/loop.py",
    "src/stage_c/pipeline.py",
    "scripts/run_stage_abc_live_pair.py",
]:
    path = PROJECT_ROOT / relative
    print("OK" if path.exists() else "MISSING", path)



## 4. 이전 with-results ZIP에서 cache만 선택적으로 추출

전체 이전 프로젝트를 다시 사용할 필요는 없다. 다음 두 경로만 추출한다.

- `data/cache/evidence_live`
- `data/raw/chembl/cache`

이전 `contexts_live`는 복사하지 않는다. v0.7.2에서 Stage A context와 local graph를
새로 만들되, 시간이 오래 걸리는 target-pair evidence 구축만 cache hit로 재사용한다.


In [ ]:

CACHE_EXTRACT_ROOT = WORK_ROOT / "old_cache_extracted"
CACHE_EXTRACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

selected_members = []

with zipfile.ZipFile(WITH_RESULTS_ZIP) as archive:
    for member in archive.infolist():
        normalized = "/" + member.filename.replace(
            "\\",
            "/",
        )

        keep = (
            "/data/cache/evidence_live/" in normalized
            or normalized.endswith(
                "/data/cache/evidence_live"
            )
            or "/data/raw/chembl/cache/" in normalized
            or normalized.endswith(
                "/data/raw/chembl/cache"
            )
        )

        if keep:
            selected_members.append(member)

    if not selected_members:
        raise FileNotFoundError(
            "with-results ZIP 안에서 evidence_live 또는 "
            "ChEMBL cache 경로를 찾지 못했습니다."
        )

    print(
        "Selected cache archive members:",
        len(selected_members),
    )

    for member in selected_members:
        archive.extract(
            member,
            CACHE_EXTRACT_ROOT,
        )

print("Selective extraction complete.")


In [ ]:

def directory_size(path: Path) -> int:
    return sum(
        item.stat().st_size
        for item in path.rglob("*")
        if item.is_file()
    )


evidence_candidates = [
    path
    for path in CACHE_EXTRACT_ROOT.rglob(
        "evidence_live"
    )
    if (
        path.is_dir()
        and (
            path
            / "pairs"
            / PAIR_KEY
        ).is_dir()
    )
]

if not evidence_candidates:
    raise FileNotFoundError(
        "이전 ZIP에서 필요한 pair cache를 찾지 못했습니다: "
        f"data/cache/evidence_live/pairs/{PAIR_KEY}"
    )

evidence_candidates.sort(
    key=directory_size,
    reverse=True,
)

OLD_EVIDENCE_ROOT = evidence_candidates[0]
NEW_EVIDENCE_ROOT = (
    PROJECT_ROOT
    / "data/cache/evidence_live"
)

NEW_EVIDENCE_ROOT.parent.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copytree(
    OLD_EVIDENCE_ROOT,
    NEW_EVIDENCE_ROOT,
    dirs_exist_ok=True,
)

print("Copied evidence cache:")
print(OLD_EVIDENCE_ROOT)
print("→")
print(NEW_EVIDENCE_ROOT)

chembl_candidates = [
    path
    for path in CACHE_EXTRACT_ROOT.rglob("cache")
    if (
        path.is_dir()
        and tuple(path.parts[-4:])
        == ("data", "raw", "chembl", "cache")
    )
]

NEW_CHEMBL_CACHE = (
    PROJECT_ROOT
    / "data/raw/chembl/cache"
)

if chembl_candidates:
    chembl_candidates.sort(
        key=directory_size,
        reverse=True,
    )

    shutil.copytree(
        chembl_candidates[0],
        NEW_CHEMBL_CACHE,
        dirs_exist_ok=True,
    )

    print("\nCopied ChEMBL raw cache:")
    print(chembl_candidates[0])
    print("→")
    print(NEW_CHEMBL_CACHE)
else:
    print(
        "\nWARNING: ChEMBL raw cache가 ZIP에 없습니다. "
        "완성된 pair cache가 있으면 현재 pair E2E는 "
        "계속 실행될 수 있습니다."
    )

# v0.7.2 context는 새로 생성
NEW_CONTEXT_ROOT = (
    PROJECT_ROOT
    / "data/cache/contexts_live"
)

if NEW_CONTEXT_ROOT.exists():
    shutil.rmtree(NEW_CONTEXT_ROOT)

NEW_CONTEXT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nFresh contexts_live:")
print(NEW_CONTEXT_ROOT)


## 5. Pair cache 완전성 검사

In [ ]:

import json

NEW_PAIR_DIR = (
    PROJECT_ROOT
    / "data/cache/evidence_live/pairs"
    / PAIR_KEY
)

required_pair_files = [
    "paired_activities.jsonl.gz",
    "aggregated_activities.jsonl.gz",
    "mmp_rules_compact.json",
    "mmp_supporting_pairs.jsonl.gz",
    "manifest.json",
]

missing = []

for filename in required_pair_files:
    path = NEW_PAIR_DIR / filename

    if path.exists():
        print(
            "OK",
            filename,
            f"{path.stat().st_size / 1024 / 1024:.2f} MB",
        )
    else:
        print("MISSING", filename)
        missing.append(filename)

if missing:
    raise RuntimeError(
        f"불완전한 pair cache입니다: {missing}"
    )

pair_manifest = json.loads(
    (
        NEW_PAIR_DIR
        / "manifest.json"
    ).read_text(encoding="utf-8")
)

print("\nPair manifest:")
print(
    json.dumps(
        pair_manifest,
        ensure_ascii=False,
        indent=2,
    )
)


## 6. v0.7.2 설치 및 Stage A/B/C import 검사

In [ ]:

import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(PROJECT_ROOT),
        "pytest",
        "pillow",
    ],
    check=True,
)

SRC_ROOT = PROJECT_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import stage_a
import stage_b
import stage_c

print("stage_a:", stage_a.__file__)
print("stage_b:", stage_b.__file__)
print("stage_c:", stage_c.__file__)


In [ ]:

from stage_a.storage.evidence_cache import (
    EvidenceCacheRepository,
)

repository = EvidenceCacheRepository(
    str(
        PROJECT_ROOT
        / "data/cache/evidence_live"
    )
)

print(
    "has_pair:",
    repository.has_pair(
        ON_TARGET,
        OFF_TARGET,
    ),
)

bundle = repository.load_pair(
    ON_TARGET,
    OFF_TARGET,
)

print("cache_hit:", bundle.cache_hit)
print("paired:", len(bundle.paired))
print("aggregated:", len(bundle.aggregated))
print("rules:", len(bundle.rules))
print("supporting pairs:", len(bundle.mmp_pairs))
print("sources:", bundle.sources)


## 7. trajectory 및 Stage C focused test

In [ ]:

if RUN_FOCUSED_TESTS:
    test_command = [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/integration/"
        "test_trajectory_mode_v072.py",
        "tests/integration/"
        "test_stage_b_stage_c_handoff.py",
        "tests/integration/"
        "test_stage_c_pipeline.py",
    ]

    subprocess.run(
        test_command,
        cwd=PROJECT_ROOT,
        env={
            **dict(__import__("os").environ),
            "PYTHONPATH": "src",
        },
        check=True,
    )
else:
    print("Focused tests skipped.")



## 8. Ollama 설치

Colab의 최신 Ollama 설치 스크립트는 압축 해제에 `zstd`가 필요할 수 있으므로
먼저 `zstd`, `curl`, `ffmpeg`를 설치한다.


In [ ]:

import os

subprocess.run(
    [
        "bash",
        "-lc",
        (
            "apt-get update -qq && "
            "apt-get install -y zstd curl ffmpeg"
        ),
    ],
    check=True,
)

if shutil.which("ollama") is None:
    subprocess.run(
        [
            "bash",
            "-lc",
            (
                "curl -fsSL "
                "https://ollama.com/install.sh | sh"
            ),
        ],
        check=True,
    )

print("Ollama binary:", shutil.which("ollama"))

subprocess.run(
    ["ollama", "--version"],
    check=True,
)


## 9. Ollama 서버 시작

In [ ]:

import time
import requests

OLLAMA_LOG = Path("/tmp/ollama_v072_server.log")


def ollama_ready() -> bool:
    try:
        response = requests.get(
            f"{OLLAMA_API_ROOT}/api/tags",
            timeout=2,
        )
        return response.ok
    except requests.RequestException:
        return False


subprocess.run(
    ["pkill", "-f", "ollama serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(1)

ollama_env = os.environ.copy()
ollama_env["OLLAMA_HOST"] = (
    "127.0.0.1:11434"
)

with OLLAMA_LOG.open(
    "w",
    encoding="utf-8",
) as log_handle:
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=ollama_env,
        start_new_session=True,
    )

for second in range(1, 121):
    if ollama_ready():
        print(
            "Ollama server ready:",
            f"{second} sec",
        )
        break

    time.sleep(1)
else:
    print(
        OLLAMA_LOG.read_text(
            encoding="utf-8",
            errors="replace",
        )[-8000:]
    )

    raise RuntimeError(
        "Ollama 서버 시작 실패"
    )


## 10. Qwen3 8B 다운로드 및 API 검사

In [ ]:

ollama_list = subprocess.run(
    ["ollama", "list"],
    text=True,
    capture_output=True,
    env=ollama_env,
    check=True,
)

print(ollama_list.stdout)

if MODEL not in ollama_list.stdout:
    subprocess.run(
        ["ollama", "pull", MODEL],
        env=ollama_env,
        check=True,
    )
else:
    print(f"{MODEL} is already installed.")


In [ ]:

api_payload = {
    "model": MODEL,
    "temperature": 0,
    "seed": 42,
    "messages": [
        {
            "role": "system",
            "content": (
                "Return only valid JSON. "
                "Do not use markdown."
            ),
        },
        {
            "role": "user",
            "content": (
                'Return {"status":"ready"} '
                "/no_think"
            ),
        },
    ],
}

api_response = requests.post(
    f"{OLLAMA_BASE_URL}/chat/completions",
    json=api_payload,
    timeout=600,
)

print("HTTP status:", api_response.status_code)

if not api_response.ok:
    print(api_response.text)
    api_response.raise_for_status()

print(
    api_response.json()
    ["choices"][0]["message"]["content"]
)

print("\nOllama process allocation:")
subprocess.run(
    ["ollama", "ps"],
    env=ollama_env,
    check=False,
)

print("\nGPU after model load:")
subprocess.run(
    ["nvidia-smi"],
    check=False,
)



## 11. 실시간 로그 실행 함수

Stage A/B/C 로그를 숨기지 않고 그대로 출력한다. 지정 시간 초과 시 해당 프로세스만
종료하고 Ollama 서버와 노트북 커널은 유지한다.


In [ ]:

import signal


def run_command_live(
    command,
    *,
    cwd,
    timeout_minutes,
    log_path,
):
    log_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("Command:")
    print(" ".join(map(str, command)))
    print()

    started = time.time()

    with log_path.open(
        "w",
        encoding="utf-8",
    ) as log_handle:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            env={
                **os.environ,
                "PYTHONPATH": "src",
                "PYTHONUNBUFFERED": "1",
            },
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            start_new_session=True,
        )

        try:
            assert process.stdout is not None

            for line in iter(
                process.stdout.readline,
                "",
            ):
                print(
                    line,
                    end="",
                    flush=True,
                )

                log_handle.write(line)
                log_handle.flush()

                if (
                    time.time() - started
                    > timeout_minutes * 60
                ):
                    raise subprocess.TimeoutExpired(
                        command,
                        timeout_minutes * 60,
                    )

            return_code = process.wait()

        except subprocess.TimeoutExpired:
            print(
                "\n[TIMEOUT] "
                f"{timeout_minutes}분 초과. "
                "현재 run을 종료합니다."
            )

            os.killpg(
                os.getpgid(process.pid),
                signal.SIGTERM,
            )

            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(
                    os.getpgid(process.pid),
                    signal.SIGKILL,
                )

            return {
                "returncode": -1,
                "timed_out": True,
                "elapsed_sec": (
                    time.time() - started
                ),
                "log_path": log_path,
            }

    return {
        "returncode": return_code,
        "timed_out": False,
        "elapsed_sec": time.time() - started,
        "log_path": log_path,
    }


## 12. Heuristic A→B→C trajectory smoke test

In [ ]:

from datetime import datetime

SESSION_TIMESTAMP = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

SMOKE_PREFIX = Path(
    "outputs/e2e_v072"
) / (
    f"{SESSION_TIMESTAMP}_"
    "heuristic_trajectory"
)

if RUN_HEURISTIC_SMOKE:
    smoke_command = [
        sys.executable,
        "-u",
        "scripts/run_stage_abc_live_pair.py",
        "--seed-smiles",
        SEED_SMILES,
        "--on-target",
        ON_TARGET,
        "--off-target",
        OFF_TARGET,
        "--heuristic",
        "--neighbor-surrogate",
        "--dynamic-discovery",
        "--search-mode",
        SEARCH_MODE,
        "--max-iterations",
        str(MAX_ITERATIONS),
        "--final-top-k",
        str(FINAL_TOP_K),
        "--output-prefix",
        str(SMOKE_PREFIX),
    ]

    smoke_info = run_command_live(
        smoke_command,
        cwd=PROJECT_ROOT,
        timeout_minutes=15,
        log_path=(
            PROJECT_ROOT
            / f"{SMOKE_PREFIX}.log"
        ),
    )

    if smoke_info["returncode"] != 0:
        raise RuntimeError(
            "Heuristic A/B/C smoke test 실패. "
            f"로그: {smoke_info['log_path']}"
        )

    print(
        "\nHeuristic E2E elapsed:",
        f"{smoke_info['elapsed_sec'] / 60:.2f} min",
    )
else:
    print("Heuristic smoke test skipped.")


## 13. Qwen3 8B 기반 실제 A→B→C trajectory 실행

In [ ]:

QWEN_PREFIX = Path(
    "outputs/e2e_v072"
) / (
    f"{SESSION_TIMESTAMP}_"
    f"{SEED_COMPOUND_ID}_"
    "qwen_trajectory"
)

qwen_command = [
    sys.executable,
    "-u",
    "scripts/run_stage_abc_live_pair.py",
    "--seed-smiles",
    SEED_SMILES,
    "--on-target",
    ON_TARGET,
    "--off-target",
    OFF_TARGET,
    "--model",
    MODEL,
    "--base-url",
    OLLAMA_BASE_URL,
    "--neighbor-surrogate",
    "--dynamic-discovery",
    "--search-mode",
    SEARCH_MODE,
    "--max-iterations",
    str(MAX_ITERATIONS),
    "--final-top-k",
    str(FINAL_TOP_K),
    "--output-prefix",
    str(QWEN_PREFIX),
]

if PREDICTION_JSON is not None:
    prediction_path = Path(
        PREDICTION_JSON
    )

    if not prediction_path.exists():
        raise FileNotFoundError(
            prediction_path
        )

    qwen_command.extend(
        [
            "--prediction-json",
            str(prediction_path),
        ]
    )

if DOCKING_JSON is not None:
    docking_path = Path(
        DOCKING_JSON
    )

    if not docking_path.exists():
        raise FileNotFoundError(
            docking_path
        )

    qwen_command.extend(
        [
            "--docking-json",
            str(docking_path),
        ]
    )

if REQUIRE_DOCKING:
    qwen_command.append(
        "--require-docking"
    )

qwen_info = run_command_live(
    qwen_command,
    cwd=PROJECT_ROOT,
    timeout_minutes=QWEN_TIMEOUT_MINUTES,
    log_path=(
        PROJECT_ROOT
        / f"{QWEN_PREFIX}.log"
    ),
)

print(
    "\nQwen E2E elapsed:",
    f"{qwen_info['elapsed_sec'] / 60:.2f} min",
)

if qwen_info["returncode"] != 0:
    print(
        OLLAMA_LOG.read_text(
            encoding="utf-8",
            errors="replace",
        )[-8000:]
    )

    raise RuntimeError(
        "Qwen A/B/C E2E 실패. "
        f"로그: {qwen_info['log_path']}"
    )


## 14. 결과 파일 로드

In [ ]:

STAGE_B_PATH = (
    PROJECT_ROOT
    / f"{QWEN_PREFIX}_stage_b.json"
)

STAGE_C_PATH = (
    PROJECT_ROOT
    / f"{QWEN_PREFIX}_stage_c.json"
)

STAGE_C_REPORT_PATH = (
    PROJECT_ROOT
    / f"{QWEN_PREFIX}_stage_c.md"
)

QWEN_LOG_PATH = (
    PROJECT_ROOT
    / f"{QWEN_PREFIX}.log"
)

for path in [
    STAGE_B_PATH,
    STAGE_C_PATH,
    STAGE_C_REPORT_PATH,
    QWEN_LOG_PATH,
]:
    print(
        "OK" if path.exists() else "MISSING",
        path,
    )

    if not path.exists():
        raise FileNotFoundError(path)

stage_b_result = json.loads(
    STAGE_B_PATH.read_text(
        encoding="utf-8"
    )
)

stage_c_result = json.loads(
    STAGE_C_PATH.read_text(
        encoding="utf-8"
    )
)

print("\nStage B status:")
print(stage_b_result["run_status"])

print("\nStage C status:")
print(stage_c_result["run_status"])


## 15. trajectory 연속성 및 누적 목적값 검증

In [ ]:

import pandas as pd

accept_steps = [
    step
    for step in stage_b_result.get(
        "trajectory",
        [],
    )
    if step.get("decision") == "ACCEPT"
]

backtrack_steps = [
    step
    for step in stage_b_result.get(
        "trajectory",
        [],
    )
    if step.get("decision") == "BACKTRACK"
]

continuity_errors = []
expected_parent = stage_b_result[
    "seed_smiles"
]

path_rows = [
    {
        "path_index": 0,
        "iteration": 0,
        "parent_smiles": None,
        "product_smiles": expected_parent,
        "step_delta_on": 0.0,
        "step_delta_s": 0.0,
        "cumulative_delta_on": 0.0,
        "cumulative_delta_s": 0.0,
        "family": "seed",
    }
]

for step in accept_steps:
    parent = step.get("parent_smiles")
    product = step.get(
        "chosen_product_smiles"
    )

    if parent != expected_parent:
        continuity_errors.append({
            "expected_parent": expected_parent,
            "recorded_parent": parent,
            "product": product,
            "iteration": step.get(
                "iteration"
            ),
        })

    path_rows.append({
        "path_index": step.get(
            "path_index"
        ),
        "iteration": step.get(
            "iteration"
        ),
        "parent_smiles": parent,
        "product_smiles": product,
        "step_delta_on": step.get(
            "predicted_delta_on"
        ),
        "step_delta_s": step.get(
            "predicted_selectivity_gain"
        ),
        "cumulative_delta_on": step.get(
            "cumulative_delta_on"
        ),
        "cumulative_delta_s": step.get(
            "cumulative_selectivity_gain"
        ),
        "family": step.get("family"),
    })

    expected_parent = product

path_df = pd.DataFrame(path_rows)

print("search_mode:", stage_b_result["search_mode"])
print("ACCEPT steps:", len(accept_steps))
print("BACKTRACK steps:", len(backtrack_steps))
print("continuity errors:", len(continuity_errors))
print(
    "active_path nodes:",
    len(
        stage_b_result.get(
            "active_path",
            [],
        )
    ),
)

if stage_b_result["search_mode"] != "trajectory":
    raise AssertionError(
        "search_mode가 trajectory가 아닙니다."
    )

if backtrack_steps:
    raise AssertionError(
        "trajectory mode에서 BACKTRACK이 기록됐습니다."
    )

if continuity_errors:
    raise AssertionError(
        "ACCEPT parent-child chain이 연속적이지 않습니다: "
        f"{continuity_errors}"
    )

display(
    path_df[
        [
            "path_index",
            "iteration",
            "step_delta_on",
            "step_delta_s",
            "cumulative_delta_on",
            "cumulative_delta_s",
            "family",
        ]
    ]
)


## 16. Stage C 최종 판정 표 및 보고서

In [ ]:

assessment_rows = []

for item in stage_c_result.get(
    "candidate_assessments",
    [],
):
    assessment_rows.append({
        "final_rank": item.get(
            "final_rank"
        ),
        "candidate_id": item.get(
            "candidate_id"
        ),
        "decision": item.get(
            "decision"
        ),
        "delta_on": item.get(
            "delta_on"
        ),
        "worst_delta_s": item.get(
            "worst_case_delta_selectivity"
        ),
        "required_off_coverage": item.get(
            "required_off_coverage"
        ),
        "independent_evidence_count": item.get(
            "independent_evidence_count"
        ),
        "evidence_score": item.get(
            "evidence_score"
        ),
        "rerank_score": item.get(
            "rerank_score"
        ),
        "decision_reasons": " | ".join(
            item.get(
                "decision_reasons",
                [],
            )
        ),
        "validation_actions": " | ".join(
            item.get(
                "validation_actions",
                [],
            )
        ),
    })

stage_c_table = pd.DataFrame(
    assessment_rows
)

print(
    "selected_candidate:",
    (
        stage_c_result
        .get("selected_candidate")
        or {}
    ).get("candidate_id"),
)

print(
    "selected_decision:",
    (
        stage_c_result
        .get("selected_candidate")
        or {}
    ).get("decision"),
)

display(stage_c_table)

STAGE_C_CSV_PATH = (
    STAGE_C_PATH.with_suffix(".csv")
)

stage_c_table.to_csv(
    STAGE_C_CSV_PATH,
    index=False,
)

print("Saved table:", STAGE_C_CSV_PATH)


In [ ]:

from IPython.display import (
    Markdown,
    display,
)

display(
    Markdown(
        STAGE_C_REPORT_PATH.read_text(
            encoding="utf-8"
        )
    )
)



## 17. 시각화 전에 결과를 즉시 Google Drive에 백업

GIF/MP4 생성 중 런타임 메모리 문제가 발생해도 Stage A/B/C 결과는 보존된다.


In [ ]:

RUN_BACKUP_DIR = (
    DRIVE_RUN_ROOT
    / (
        f"{SESSION_TIMESTAMP}_"
        f"{SEED_COMPOUND_ID}_"
        "trajectory_e2e"
    )
)

RUN_BACKUP_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

result_files = [
    STAGE_B_PATH,
    STAGE_C_PATH,
    STAGE_C_REPORT_PATH,
    STAGE_C_CSV_PATH,
    QWEN_LOG_PATH,
    OLLAMA_LOG,
]

if RUN_HEURISTIC_SMOKE:
    result_files.extend([
        PROJECT_ROOT
        / f"{SMOKE_PREFIX}_stage_b.json",
        PROJECT_ROOT
        / f"{SMOKE_PREFIX}_stage_c.json",
        PROJECT_ROOT
        / f"{SMOKE_PREFIX}_stage_c.md",
        PROJECT_ROOT
        / f"{SMOKE_PREFIX}.log",
    ])

run_config = {
    "created_at": datetime.now().isoformat(),
    "project_version": "0.7.1",
    "seed_compound_id": SEED_COMPOUND_ID,
    "seed_smiles": SEED_SMILES,
    "on_target": ON_TARGET,
    "off_target": OFF_TARGET,
    "model": MODEL,
    "search_mode": SEARCH_MODE,
    "max_iterations": MAX_ITERATIONS,
    "final_top_k": FINAL_TOP_K,
    "prediction_json": (
        str(PREDICTION_JSON)
        if PREDICTION_JSON is not None
        else None
    ),
    "docking_json": (
        str(DOCKING_JSON)
        if DOCKING_JSON is not None
        else None
    ),
    "stage_b_status": stage_b_result.get(
        "run_status"
    ),
    "stage_c_status": stage_c_result.get(
        "run_status"
    ),
    "accept_steps": len(accept_steps),
    "backtrack_steps": len(
        backtrack_steps
    ),
}

(
    RUN_BACKUP_DIR
    / "run_config.json"
).write_text(
    json.dumps(
        run_config,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

copied_files = []

for source in result_files:
    if source.exists():
        destination = (
            RUN_BACKUP_DIR
            / source.name
        )

        shutil.copy2(
            source,
            destination,
        )

        copied_files.append(
            destination
        )

subprocess.run(
    ["sync"],
    check=False,
)

print("Immediate Drive backup complete:")
print(RUN_BACKUP_DIR)

for path in copied_files:
    print(" -", path.name)


## 18. 시각화 전 RAM 정리

In [ ]:

import gc

subprocess.run(
    ["ollama", "stop", MODEL],
    env=ollama_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for variable_name in [
    "bundle",
    "repository",
]:
    globals().pop(
        variable_name,
        None,
    )

gc.collect()

subprocess.run(
    ["free", "-h"],
    check=False,
)



## 19. 실제 누적 목적공간 trajectory 생성

- x축: 최초 seed 대비 누적 on-target pActivity 변화
- y축: 최초 seed 대비 누적 선택성 변화
- 회색 점: full EGFR–HER2 실측 화합물
- 별표: 실제 `active_path`
- 초록 영역: `Δon ≥ -1.0`, `ΔS ≥ +0.1`
- trajectory mode이므로 별은 이전 seed로 돌아가지 않는다.


In [ ]:

import gzip
import numpy as np
import matplotlib.pyplot as plt

PAIRED_PATH = (
    NEW_PAIR_DIR
    / "paired_activities.jsonl.gz"
)

paired_records = []

with gzip.open(
    PAIRED_PATH,
    "rt",
    encoding="utf-8",
) as handle:
    for line in handle:
        line = line.strip()

        if line:
            paired_records.append(
                json.loads(line)
            )

paired_df = pd.DataFrame(
    paired_records
)

baseline = stage_b_result["baseline"]
seed_p_on = float(
    baseline["p_activity_on"]
)

seed_selectivity = float(
    baseline["selectivity_S"][
        OFF_TARGET
    ]
)

paired_df["objective_x"] = (
    paired_df["p_on"].astype(float)
    - seed_p_on
)

paired_df["objective_y"] = (
    paired_df[
        "selectivity"
    ].astype(float)
    - seed_selectivity
)

objective_path = [
    {
        "path_index": 0,
        "iteration": 0,
        "x": 0.0,
        "y": 0.0,
        "smiles": stage_b_result[
            "seed_smiles"
        ],
    }
]

for step in accept_steps:
    cumulative_x = step.get(
        "cumulative_delta_on"
    )
    cumulative_y = step.get(
        "cumulative_selectivity_gain"
    )

    if (
        cumulative_x is None
        or cumulative_y is None
    ):
        raise RuntimeError(
            "v0.7.2 trajectory step에 누적 "
            "objective 값이 없습니다."
        )

    objective_path.append({
        "path_index": step.get(
            "path_index"
        ),
        "iteration": step.get(
            "iteration"
        ),
        "x": float(cumulative_x),
        "y": float(cumulative_y),
        "smiles": step.get(
            "chosen_product_smiles"
        ),
    })

objective_df = pd.DataFrame(
    objective_path
)

display(objective_df)


In [ ]:

VIS_DIR = (
    PROJECT_ROOT
    / "outputs/e2e_v072"
    / (
        f"{SESSION_TIMESTAMP}_"
        "objective_visualization"
    )
)

VIS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OBJECTIVE_CSV_PATH = (
    VIS_DIR
    / "trajectory_objective_points.csv"
)

OBJECTIVE_STATIC_PATH = (
    VIS_DIR
    / "trajectory_objective_space.png"
)

OBJECTIVE_GIF_PATH = (
    VIS_DIR
    / "trajectory_objective_space.gif"
)

OBJECTIVE_MP4_PATH = (
    VIS_DIR
    / "trajectory_objective_space.mp4"
)

objective_df.to_csv(
    OBJECTIVE_CSV_PATH,
    index=False,
)

X_THRESHOLD = -1.0
Y_THRESHOLD = 0.1

x_path = objective_df[
    "x"
].to_numpy(dtype=float)

y_path = objective_df[
    "y"
].to_numpy(dtype=float)

x_min = min(
    X_THRESHOLD - 0.2,
    x_path.min() - 0.35,
)

x_max = max(
    0.5,
    x_path.max() + 0.35,
)

y_min = min(
    -0.3,
    y_path.min() - 0.35,
)

y_max = max(
    0.8,
    y_path.max() + 0.35,
)

# movement가 작아 보이지 않도록 path 중심으로 확대하되
# gate 기준선은 반드시 포함한다.
local_background = paired_df[
    paired_df["objective_x"].between(
        x_min,
        x_max,
    )
    & paired_df["objective_y"].between(
        y_min,
        y_max,
    )
].copy()

print(
    "Local measured background points:",
    len(local_background),
)


In [ ]:

def draw_objective_base(ax):
    ax.scatter(
        local_background[
            "objective_x"
        ],
        local_background[
            "objective_y"
        ],
        s=16,
        alpha=0.20,
        label=(
            "Measured EGFR–HER2 "
            "compounds"
        ),
    )

    ax.fill_between(
        [X_THRESHOLD, x_max],
        Y_THRESHOLD,
        y_max,
        alpha=0.08,
        label="Feasible region",
    )

    ax.axvline(
        X_THRESHOLD,
        linestyle="--",
        linewidth=1.4,
        label="On-target drop limit",
    )

    ax.axhline(
        Y_THRESHOLD,
        linestyle=":",
        linewidth=1.4,
        label=(
            "Minimum selectivity gain"
        ),
    )

    ax.axvline(
        0.0,
        linewidth=0.8,
        alpha=0.35,
    )

    ax.axhline(
        0.0,
        linewidth=0.8,
        alpha=0.35,
    )

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    ax.set_xlabel(
        "Cumulative Δ on-target "
        "pActivity vs seed"
    )

    ax.set_ylabel(
        "Cumulative Δ selectivity "
        "vs seed"
    )

    ax.grid(alpha=0.12)


fig, ax = plt.subplots(
    figsize=(10, 7)
)

draw_objective_base(ax)

for index in range(
    len(objective_df) - 1
):
    start = objective_df.iloc[
        index
    ][["x", "y"]].to_numpy(
        dtype=float
    )

    end = objective_df.iloc[
        index + 1
    ][["x", "y"]].to_numpy(
        dtype=float
    )

    ax.annotate(
        "",
        xy=end,
        xytext=start,
        arrowprops={
            "arrowstyle": "->",
            "linewidth": 2.5,
            "connectionstyle": (
                "arc3,rad=0.06"
            ),
        },
    )

for _, row in objective_df.iterrows():
    ax.scatter(
        [row["x"]],
        [row["y"]],
        marker="*",
        s=380,
        edgecolors="black",
        linewidths=1.1,
        zorder=6,
    )

    ax.annotate(
        (
            f"S{int(row['path_index'])}\n"
            f"Δon={row['x']:+.2f}, "
            f"ΔS={row['y']:+.2f}"
        ),
        (row["x"], row["y"]),
        xytext=(7, 8),
        textcoords="offset points",
        fontsize=9,
    )

ax.set_title(
    "v0.7.2 trajectory in "
    "selectivity objective space"
)

ax.legend(
    loc="best",
    fontsize=8,
)

fig.tight_layout()

fig.savefig(
    OBJECTIVE_STATIC_PATH,
    dpi=180,
    facecolor="white",
)

plt.show()
plt.close(fig)

print("Saved:", OBJECTIVE_STATIC_PATH)


## 20. 별표가 실제 step마다 전진하는 GIF/MP4

In [ ]:

import tempfile

FRAMES_PER_MOVE = 22
PAUSE_FRAMES = 7
FPS = 12

timeline = []

start_xy = objective_df.iloc[
    0
][["x", "y"]].to_numpy(
    dtype=float
)

for _ in range(PAUSE_FRAMES):
    timeline.append({
        "xy": start_xy.copy(),
        "completed_segments": 0,
        "label": "Seed molecule",
    })

for segment_index in range(
    len(objective_df) - 1
):
    start = objective_df.iloc[
        segment_index
    ][["x", "y"]].to_numpy(
        dtype=float
    )

    end = objective_df.iloc[
        segment_index + 1
    ][["x", "y"]].to_numpy(
        dtype=float
    )

    for fraction in np.linspace(
        0.0,
        1.0,
        FRAMES_PER_MOVE,
    ):
        point = (
            (1.0 - fraction) * start
            + fraction * end
        )

        timeline.append({
            "xy": point,
            "completed_segments": (
                segment_index
            ),
            "label": (
                f"S{segment_index} → "
                f"S{segment_index + 1}"
            ),
        })

    for _ in range(PAUSE_FRAMES):
        timeline.append({
            "xy": end.copy(),
            "completed_segments": (
                segment_index + 1
            ),
            "label": (
                f"State "
                f"S{segment_index + 1} · "
                f"Δon={end[0]:+.2f}, "
                f"ΔS={end[1]:+.2f}"
            ),
        })

# ACCEPT가 없어도 seed-only GIF를 생성한다.
if len(objective_df) == 1:
    for _ in range(PAUSE_FRAMES * 2):
        timeline.append({
            "xy": start_xy.copy(),
            "completed_segments": 0,
            "label": (
                "No accepted child · "
                "local optimum/abstention"
            ),
        })

print("Animation frames:", len(timeline))

FRAME_DIR = Path(
    tempfile.mkdtemp(
        prefix="fos_v072_objective_"
    )
)


In [ ]:

def render_objective_frame(
    frame,
    frame_index,
):
    fig, ax = plt.subplots(
        figsize=(10, 7)
    )

    draw_objective_base(ax)

    completed = frame[
        "completed_segments"
    ]

    for segment_index in range(
        completed
    ):
        start = objective_df.iloc[
            segment_index
        ][["x", "y"]].to_numpy(
            dtype=float
        )

        end = objective_df.iloc[
            segment_index + 1
        ][["x", "y"]].to_numpy(
            dtype=float
        )

        ax.annotate(
            "",
            xy=end,
            xytext=start,
            arrowprops={
                "arrowstyle": "->",
                "linewidth": 2.5,
                "connectionstyle": (
                    "arc3,rad=0.06"
                ),
                "alpha": 0.8,
            },
        )

    visited_count = min(
        completed + 1,
        len(objective_df),
    )

    for row_index in range(
        visited_count
    ):
        row = objective_df.iloc[
            row_index
        ]

        ax.scatter(
            [row["x"]],
            [row["y"]],
            marker="*",
            s=200,
            edgecolors="black",
            linewidths=0.8,
            zorder=5,
        )

    current_x, current_y = frame["xy"]

    ax.scatter(
        [current_x],
        [current_y],
        marker="*",
        s=720,
        edgecolors="black",
        linewidths=1.6,
        zorder=8,
    )

    ax.scatter(
        [current_x],
        [current_y],
        marker="o",
        s=1150,
        facecolors="none",
        linewidths=2.5,
        zorder=7,
    )

    ax.set_title(
        "v0.7.2 agent movement in "
        "selectivity objective space\n"
        + frame["label"]
    )

    ax.legend(
        loc="best",
        fontsize=8,
    )

    fig.tight_layout()

    frame_path = (
        FRAME_DIR
        / f"frame_{frame_index:05d}.png"
    )

    # bbox_inches='tight'를 쓰지 않아
    # 모든 프레임 크기를 동일하게 유지한다.
    fig.savefig(
        frame_path,
        dpi=100,
        facecolor="white",
    )

    plt.close(fig)
    gc.collect()


for frame_index, frame in enumerate(
    timeline
):
    render_objective_frame(
        frame,
        frame_index,
    )

    if (
        frame_index % 10 == 0
        or frame_index
        == len(timeline) - 1
    ):
        print(
            f"Rendered "
            f"{frame_index + 1}/"
            f"{len(timeline)}"
        )


In [ ]:

input_pattern = str(
    FRAME_DIR / "frame_%05d.png"
)

ffmpeg_mp4 = [
    "ffmpeg",
    "-y",
    "-framerate",
    str(FPS),
    "-start_number",
    "0",
    "-i",
    input_pattern,
    "-vf",
    (
        "pad=ceil(iw/2)*2:"
        "ceil(ih/2)*2"
    ),
    "-c:v",
    "libx264",
    "-pix_fmt",
    "yuv420p",
    "-movflags",
    "+faststart",
    str(OBJECTIVE_MP4_PATH),
]

subprocess.run(
    ffmpeg_mp4,
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

palette_path = (
    FRAME_DIR / "palette.png"
)

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-framerate",
        str(FPS),
        "-start_number",
        "0",
        "-i",
        input_pattern,
        "-vf",
        (
            "pad=ceil(iw/2)*2:"
            "ceil(ih/2)*2,"
            "fps=12,"
            "scale=900:-2:"
            "flags=lanczos,"
            "palettegen=max_colors=128"
        ),
        str(palette_path),
    ],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-framerate",
        str(FPS),
        "-start_number",
        "0",
        "-i",
        input_pattern,
        "-i",
        str(palette_path),
        "-lavfi",
        (
            "[0:v]"
            "pad=ceil(iw/2)*2:"
            "ceil(ih/2)*2,"
            "fps=12,"
            "scale=900:-2:"
            "flags=lanczos[x];"
            "[x][1:v]"
            "paletteuse="
            "dither=bayer:"
            "bayer_scale=3"
        ),
        "-loop",
        "0",
        str(OBJECTIVE_GIF_PATH),
    ],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

shutil.rmtree(
    FRAME_DIR,
    ignore_errors=True,
)

gc.collect()

print("MP4:", OBJECTIVE_MP4_PATH)
print("GIF:", OBJECTIVE_GIF_PATH)


In [ ]:

from IPython.display import (
    Image as DisplayImage,
    Video,
    display,
)

display(
    Video(
        str(OBJECTIVE_MP4_PATH),
        embed=True,
        width=900,
    )
)

display(
    DisplayImage(
        filename=str(
            OBJECTIVE_GIF_PATH
        ),
        width=900,
    )
)


## 21. 시각화와 전체 결과를 Drive 백업에 추가

In [ ]:

visualization_files = [
    OBJECTIVE_CSV_PATH,
    OBJECTIVE_STATIC_PATH,
    OBJECTIVE_GIF_PATH,
    OBJECTIVE_MP4_PATH,
]

for source in visualization_files:
    if source.exists():
        shutil.copy2(
            source,
            RUN_BACKUP_DIR
            / source.name,
        )

# 로컬 결과 묶음도 ZIP으로 생성
LOCAL_BUNDLE_BASE = (
    PROJECT_ROOT
    / "outputs/e2e_v072"
    / (
        f"{SESSION_TIMESTAMP}_"
        f"{SEED_COMPOUND_ID}_"
        "complete_bundle"
    )
)

LOCAL_BUNDLE_PATH = Path(
    shutil.make_archive(
        str(LOCAL_BUNDLE_BASE),
        "zip",
        root_dir=RUN_BACKUP_DIR,
    )
)

shutil.copy2(
    LOCAL_BUNDLE_PATH,
    RUN_BACKUP_DIR
    / LOCAL_BUNDLE_PATH.name,
)

subprocess.run(
    ["sync"],
    check=False,
)

print("Final Drive backup:")
print(RUN_BACKUP_DIR)

print("\nFiles:")
for path in sorted(
    RUN_BACKUP_DIR.iterdir()
):
    if path.is_file():
        print(
            path.name,
            f"{path.stat().st_size / 1024 / 1024:.2f} MB",
        )



## 22. 최종 판정

### 정상적인 multi-step trajectory

```text
ACCEPT steps >= 2
BACKTRACK steps = 0
continuity errors = 0
```

objective GIF에서 별이 `S0 → S1 → S2 ...`로 전진한다.

### 한 단계에서 종료

```text
ACCEPT steps = 1
BACKTRACK steps = 0
```

구현 실패가 아니라 첫 ACCEPT 후보에서 통과 가능한 자식이 없어
`local_optimum`으로 종료된 것이다.

### 후보를 하나도 채택하지 않음

```text
ACCEPT steps = 0
```

seed에서 `no_safe_candidate`, `needs_validation` 또는 근거 부족으로 중단한 것이다.
Stage C는 upstream abstention을 그대로 유지한다.

### Stage C가 NEEDS_VALIDATION

독립 QSAR/docking provider를 넣지 않았다면 Stage A/B와 같은 cache에서 나온
MMP·neighbor 근거만으로 `SUPPORTED`로 승격하지 않는 보수적 동작이다.
